# jitools tutorial

jitools is a Python library for working with **just intonation (JI)** — a tuning system where every interval is expressed as a ratio of small integers. The library represents pitches as **monzos** (prime-exponent vectors), computes Helmholtz-Ellis JI (HEJI2) notation, and finds enharmonic equivalents across the prime-limit spectrum.

This notebook covers:
1. Creating a `Pitch`
2. Inspecting pitch properties
3. Working with `PitchCollection`
4. Changing the reference pitch
5. Finding enharmonic equivalents
6. Generating a custom lookup table

In [1]:
from fractions import Fraction
from jitools import Pitch, PitchCollection

---
## 1. Creating a Pitch

A `Pitch` represents a frequency ratio relative to a reference pitch (default: A4 = 440 Hz). You can define the ratio three ways.

In [2]:
# From a ratio as a (numerator, denominator) tuple
p = Pitch(p=(3, 2))
print(p.ratio)               # Fraction(3, 2)
print(p.freq)                # frequency in Hz (3/2 of 440)
print(p.distance_in_cents_from_reference)  # ~702 cents above A4

3/2
660.0
701.955000865388


In [3]:
# From a Python Fraction
p = Pitch(p=Fraction(7, 4))
print(p.ratio)

7/4


In [4]:
# From a monzo (prime-exponent vector)
# Primes are indexed as [2, 3, 5, 7, 11, 13, ...]
# [-1, 1] means 2^-1 * 3^1 = 3/2
p = Pitch(p=[-1, 1])
print(p.ratio)    # Fraction(3, 2)
print(p.monzo)    # stored internally as a trimmed list

# [-2, 0, 1] means 2^-2 * 5^1 = 5/4
p2 = Pitch(p=[-2, 0, 1])
print(p2.ratio)   # Fraction(5, 4)

3/2
[-1, 1]
5/4


---
## 2. Inspecting pitch properties

In [2]:
p = Pitch(p=(7, 4))   # harmonic seventh

print("ratio:              ", p.ratio)
print("monzo:              ", p.monzo)
print("constituent primes: ", p.constituent_primes)
print("freq (Hz):          ", round(p.freq, 3))
print("MIDI key number:    ", round(p.keynum, 3))
print("cents from ref:     ", round(p.distance_in_cents_from_reference, 4))
print("harmonic distance:  ", round(p.harmonic_distance, 4))

ratio:               7/4
monzo:               [-2, 0, 0, 1]
constituent primes:  [2, 7]
freq (Hz):           770.0
MIDI key number:     78.688
cents from ref:      968.8259
harmonic distance:   4.8074


In [6]:
# HEJI2 notation
print("accidental string:  ", p.accidental_string)
print("letter name:        ", p.letter_name)
print("num symbols:        ", p.num_symbols)
print("letter+octave+cents:", p.letter_name_and_octave_and_cents)

accidental string:   <
letter name:         G
num symbols:         1
letter+octave+cents: G5 -31.17409


In [7]:
# Normalized form — ratio shifted into [1, 2), i.e. within one octave above 1/1
p = Pitch(p=(1, 3))   # a fifth below
print("ratio:             ", p.ratio)
print("normalized ratio:  ", p.normalized_ratio)
print("normalized monzo:  ", p.normalized_monzo)

ratio:              1/3
normalized ratio:   4/3
normalized monzo:   [2, -1]


In [8]:
# print_info() gives a readable summary
Pitch(p=(5, 4)).print_info()


BASIC INFO
ratio: 5/4
monzo: [-2, 0, 1]
constituent primes: [2, 5]
frequency (Hz): 550.0
MIDI key number: 72.86314
distance from 1/1 (cents): 386.31371
harmonic distance: 4.32193
Helmholtz-Ellis notation (text string, letter name): ('u', 'C')
12-ED2 pitch and cent deviation: C#/Db5 -13.68629



---
## 3. Working with PitchCollection

A `PitchCollection` takes a list of ratios and computes harmonic relationships across the whole chord: intervals, resultant tones, tuneability, harmonic distance, and more.

In [4]:
# Just major triad: 1/1, 5/4, 3/2
triad = PitchCollection(pc=[(1, 1), (5, 4), (3, 2)])

print("ratios:              ", [str(r) for r in triad.ratios])
print("constituent primes:  ", triad.constituent_primes)
print("harmonics:           ", triad.harmonics)     # e.g. [4, 5, 6]
print("least common partial:", triad.least_common_partial)
print("periodicity pitch:   ", round(triad.periodicity_pitch, 2), "Hz")
print("HD sum:              ", round(triad.hd_sum, 4))
print("HD average:          ", round(triad.hd_avg, 4))

ratios:               ['1', '5/4', '3/2']
constituent primes:   [2, 3, 5]
harmonics:            [4, 5, 6]
least common partial: 60
periodicity pitch:    110.0 Hz
HD sum:               6.9069
HD average:           2.3023


In [5]:
# Intervals between all pitch pairs
print("all intervals:")
for i in triad.intervals:
    print(" ", i)

print("\ntuneable intervals (Sabat-Schweinitz):")
for i in triad.tuneable_intervals:
    print(" ", i)

all intervals:
  6/5
  5/4
  3/2

tuneable intervals (Sabat-Schweinitz):
  6/5
  5/4
  3/2


In [5]:
# Difference and summation tones
print("difference tones:         ", [str(r) for r in triad.difference_tones])
print("tuneable difference tones:", [str(r) for r in triad.tuneable_difference_tones])
print("summation tones:          ", [str(r) for r in triad.summation_tones])

difference tones:          ['1/4', '1/2']
tuneable difference tones: ['1/4', '1/2']
summation tones:           ['9/4', '5/2', '11/4']


In [15]:
# HEJI notations for each pitch in the collection
for acc, letter in triad.notations:
    print(f"  {letter}  {acc}")

  A  n
  C  u
  E  n


In [6]:
# print_info() gives a structured overview of the whole collection
triad.print_info()


BASIC INFO
ratios: ['1/1', '5/4', '3/2']
frequencies (Hz): ['440.0', '550.0', '660.0']
MIDI key numbers: ['69.0', '72.86314', '76.01955']
Helmholtz-Ellis notations (text string, letter name): [('n', 'A'), ('u', 'C'), ('n', 'E')]
12-ED2 pitch and cent deviations: ['A4 +0.0', 'C#/Db5 -13.68629', 'E5 +1.955']
harmonic constellation: 4:5:6
sequential intervals: ['5/4', '6/5']
normalized ratios: ['1/1', '5/4', '3/2']
inversion: ['1/1', '6/5', '3/2']



In [17]:
# Compare a 7-limit seventh chord
seventh = PitchCollection(pc=[(1, 1), (5, 4), (3, 2), (7, 4)])
print("harmonics:    ", seventh.harmonics)      # e.g. [4, 5, 6, 7]
print("HD sum:       ", round(seventh.hd_sum, 4))
print("constituent:  ", seventh.constituent_primes)

harmonics:     [4, 5, 6, 7]
HD sum:        11.7142
constituent:   [2, 3, 5, 7]


### Sorting, transposing, and updating

In [7]:
# sort_by re-orders pitches in the collection
col = PitchCollection(pc=[(3, 2), (1, 1), (5, 4)])
print("before sort:", [str(r) for r in col.ratios])
col.sort_by("ratios")
print("after sort: ", [str(r) for r in col.ratios])

before sort: ['1', '5/4', '3/2']
after sort:  ['1', '5/4', '3/2']


In [8]:
# transpose shifts every pitch by a given interval
col = PitchCollection(pc=[(1, 1), (5, 4), (3, 2)])
print("before transpose:", [str(r) for r in col.ratios])
col.transpose((2, 3))   # down a fifth
print("after transpose: ", [str(r) for r in col.ratios])

before transpose: ['1', '5/4', '3/2']
after transpose:  ['2/3', '5/6', '1']


In [10]:
# update() re-initializes with new parameters
p = Pitch(p=(3, 2))
print(p.ratio)
p.update(p=(5, 4))
print(p.ratio)

3/2
5/4


---
## 4. Changing the reference pitch

All pitches are defined relative to a reference (`rp`, default `"A4"`) at a given frequency (`rf`, default `440.0` Hz).

In [11]:
# 3/2 above C4 (at 261.63 Hz)
p = Pitch(p=(3, 2), rp="C4", rf=261.63)
print("freq:   ", round(p.freq, 2), "Hz")
print("letter: ", p.letter_name)
print("cents:  ", round(p.distance_in_cents_from_reference, 2))

freq:    392.44 Hz
letter:  G
cents:   701.96


In [12]:
# PitchCollection inherits the same reference for all pitches
col = PitchCollection(pc=[(1, 1), (5, 4), (3, 2)], rp="C4", rf=261.63)
print("freqs:", [round(f, 2) for f in col.freqs])

freqs: [261.63, 327.04, 392.44]


---
## 5. Finding enharmonic equivalents

`get_enharmonics()` searches a prebuilt lookup table for JI intervals whose pitch class is within `tolerance` cents of the input pitch. This is useful for finding simpler or differently-notated versions of the same (or nearby) pitch.

In [3]:
p = Pitch(p=(81, 64))   # Pythagorean major third

enharmonics = p.get_enharmonics(
    tolerance=5,        # search within ±5 cents
    max_symbols=2,      # at most 2 accidental characters
    limit=13,           # only use primes up to 13
    max_hd=20,          # harmonic distance cutoff
    max_candidates=8
)

for candidate in enharmonics:
    print(candidate)

In [10]:
# print_enharmonics_info() formats the same results for readability
p.print_enharmonics_info(tolerance=5, max_symbols=2, limit=13, max_hd=20)

In [11]:
# Exclude specific primes from candidates
p = Pitch(p=(11, 8))   # 11th harmonic
p.print_enharmonics_info(
    tolerance=3,
    max_symbols=2,
    exclude_primes=[7],   # no septimal intervals
    limit=13
)

---
## 6. Generating a custom lookup table

`get_enharmonics()` searches a prebuilt CSV table that lists every JI interval with a valid HEJI2 accidental notation, sorted by pitch class. The default table ships with the library and covers all intervals up to 3 accidental symbols across primes 2–47.

You can generate a custom table to change the scope of the search:

- **`max_symbols`** — the maximum number of accidental characters. The default is 3. Increasing this adds more complex (higher-prime or multi-comma) notations; decreasing it gives a smaller, faster table limited to simpler accidentals.
- **`max_prime_3`** — how far along the chain of perfect fifths to search (default ±70). Rarely needs changing.
- **`max_prime_5`** — the maximum number of syntonic comma arrows (default ±4, the HEJI hard limit).

Once generated, pass the file path to `get_enharmonics()` via `lookup_table_path`.

In [13]:
from jitools import generate_enharmonic_lookup_table

# Generate a table limited to 2-symbol accidentals
# workers=1 avoids subprocess spawning (safer in notebooks)
results = generate_enharmonic_lookup_table(
    max_symbols=2,
    max_prime_3=70,
    max_prime_5=4,
    output_path="/tmp/my_lookup_table.csv",
    workers=1,
    verbose=True
)
print(f"{len(results):,} entries")

  27 templates, 34,263 candidates, 1 workers
  8,757 entries in 1.1s
8,757 entries


In [14]:
from jitools import Pitch

# Use the custom table for an enharmonics search
p = Pitch(p=(81, 64))
p.print_enharmonics_info(
    tolerance=2,
    max_symbols=2,
    lookup_table_path="/tmp/my_lookup_table.csv"
)